# 🚀✨ Spaceship Titanic
_A data science project to predict [which passengers are transported to an alternate dimension](https://www.kaggle.com/competitions/spaceship-titanic/overview)_

---

### **Created by: Antonio Kevin**
🌐 [**Kaggle**](https://www.kaggle.com/akkevin) | 💼 [**LinkedIn**](https://www.linkedin.com/in/antonio-kevin/) | 🧑‍💻 [**GitHub**](https://github.com/akkevinn)

---

## **Table of Contents**
1. [Problem Understanding](#Problem-Understanding)
2. [Approach](#Approach)
3. [Data Preprocessing](#Data-Preprocessing)
4. [Feature Engineering](#Feature-Engineering)
5. [Model Training](#Model-Training)
6. [Prediction & Submission](#Prediction-and-Submission)

---

## **Problem Understanding**

### **Business Objective**:
The Spaceship Titanic has encountered a catastrophic spacetime anomaly, resulting in nearly half its passengers being transported to an alternate dimension. Our mission is to build a predictive model that determines whether a given passenger was transported or not, based on the available passenger records recovered from the damaged system.
Successfully predicting passenger outcomes will:
- **Aid rescue operations** in locating and recovering transported individuals.
- **Improve future anomaly detection** and **emergency response protocols**.
- **Enhance understanding** of factors contributing to dimensional transport events.

### **Evaluation Metric**:
Given the critical nature of the task — ensuring no transported passengers are overlooked while avoiding false identifications — we will evaluate the model based on the following metrics:

- **Accuracy**:  
  Measures the overall correctness of predictions. Useful when classes are balanced.

- **Precision** (Weighted):  
  Indicates the proportion of true transported passengers among those predicted as transported. Important for minimizing false positives.

- **Recall** (Weighted):  
  Measures the proportion of actual transported passengers correctly identified. Critical to minimize false negatives (missing a transported passenger).

- **F1-Score** (Weighted):  
  The harmonic mean of precision and recall. Especially valuable if there's a trade-off between precision and recall.

- **ROC AUC Score**:  
  Measures the model's ability to distinguish between transported and non-transported passengers across all classification thresholds. Useful for evaluating overall model discrimination ability.

---

### **Result**
- Score = 0.80243.
- Leaderboard (as of 28 Apr 2025) = Top 20% (out of 2197 teams).

## **Approach**
This notebook will walk through the following steps:

1. **Data Preprocessing**: Clean and prepare the data for modeling.
2. **Feature Engineering**: Create meaningful features for model training.
3. **Model Training**: Train and fine-tune a suitable model for the prediction task.
4. **Evaluation**: Assess model performance using F1-Score.
5. **Prediction on Test Data and Submission**:
    - Use the trained model to make predictions on the test dataset.
    - Format the predictions for submission according to the competition requirements.
    - Save the results to a CSV file for submission.

---

In [1]:
!pip install optuna
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
import optuna
from itertools import product
from sklearn.metrics import (accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score)
import sklearn
sklearn.set_config(transform_output="pandas")

# Mount the drive to access files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Data Preprocessing**

In [2]:
# Load data
dir_path = '/content/drive/My Drive/Colab Notebooks/Spaceship Titanic/'
file_path = dir_path + 'train.csv'
train_data = pd.read_csv(file_path)

file_path = dir_path + 'test.csv'
test_data = pd.read_csv(file_path)

print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")

Train shape: (8693, 14), Test shape: (4277, 13)


In [110]:
# Convert Transported into int
train_data['Transported'] = train_data['Transported'].astype(int)

train_data['dataset'] = 'train'
test_data['dataset'] = 'test'

data = pd.concat([train_data, test_data]).reset_index(drop=True)
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,dataset
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,0.0,train
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,1.0,train
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,0.0,train
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,0.0,train
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,1.0,train


In [111]:
# Parse Cabin = deck/num/side
data['cabin_deck'] = data['Cabin'].str.split('/').str[0]
data['cabin_num'] = pd.to_numeric(data['Cabin'].str.split('/').str[1])
data['cabin_side'] = data['Cabin'].str.split('/').str[2]

# Parse first & last name
data['last_name'] = test['Name'].str.split().str[-1]
data['first_name'] = test['Name'].str.split().str[0]

In [112]:
data.isna().sum()

,0
PassengerId,0
HomePlanet,288
CryoSleep,310
Cabin,299
Destination,274
Age,270
VIP,296
RoomService,263
FoodCourt,289
ShoppingMall,306


In [113]:
# Handle Missing Values

# Define numerical and categorical columns
numerical_col_list = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
categorical_col_list = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'cabin_side', 'Name']

# Fill numerical columns with median
for col in numerical_col_list:
    data[col] = data[col].fillna(data[col].median())

# Fill categorical columns with mode
for col in categorical_col_list:
    data[col] = data[col].fillna(data[col].mode()[0])

# Cabin Deck: fill with mode by HomePlanet & Destination
data['cabin_deck'] = data.groupby(['HomePlanet', 'Destination'])['cabin_deck'].transform(lambda x: x.fillna(x.mode()[0]))

# Cabin Num: fill with median by HomePlanet & cabin_deck
data['cabin_num'] = data.groupby(['HomePlanet', 'cabin_deck'])['cabin_num'].transform(lambda x: x.fillna(x.median()))

<ipython-input-113-5772afa5a288>:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])


## **Feature Engineering**

In [114]:
# Feature Engineering

# Caclulate total expenses
data['total_expenses'] = data[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)

# Passenger group count
data['passenger_group'] = data['PassengerId'].str.split('_').str[0]
data['passenger_group_count'] = data.groupby('passenger_group')['PassengerId'].transform('count')

# Calculate probability of CryoSleep per each category
category_list = ['VIP', 'passenger_group_count', 'HomePlanet', 'Destination', 'cabin_deck', 'cabin_side']
for col in category_list:
  data['proba_by_' + col] = data.groupby(col)['CryoSleep'].transform('mean')

# Calculate avg expenses per each category
for col in category_list:
  data['avg_expense_by_' + col] = data.groupby(col)['total_expenses'].transform('mean')

# # Combine features
# data['cabin_deck_side'] = data['cabin_deck'] + '_' + data['cabin_side']
# data['cabin_deck_CryoSleep'] = data['cabin_deck'] + '_' + data['CryoSleep'].astype(str)
# data['cabin_side_CryoSleep'] = data['cabin_side'] + '_' + data['CryoSleep'].astype(str)
# data['cabin_deck_side_CryoSleep'] = (
#     data['cabin_deck'] + '_' +
#     data['cabin_side'] + '_' +
#     data['CryoSleep'].astype(str)
# )

# One-hot encoding: HomePlanet, Destination, cabin_deck, cabin_side
columns = ['HomePlanet', 'Destination', 'cabin_deck', 'cabin_side']
prefix = ['homeplanet', 'destination', 'deck', 'side']
# columns = ['HomePlanet', 'Destination', 'cabin_deck', 'cabin_side',
#            'cabin_deck_side', 'cabin_deck_CryoSleep', 'cabin_side_CryoSleep',
#            'cabin_deck_side_CryoSleep']
# prefix = ['homeplanet', 'destination', 'deck', 'side', 'deck_side', 'deck_cryo', 'side_cryo', 'deck_side_cryo']
data = pd.get_dummies(data, columns=columns, prefix=prefix)

# Last name frequency
# data['last_name_count'] = data.groupby('last_name')['last_name'].transform('count')
# data['is_alone_last_name'] = (data['last_name_count'] == 1).astype(int)

In [115]:
# Features
feature_list = ['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt',
                'ShoppingMall', 'Spa', 'VRDeck',
                'total_expenses', 'passenger_group_count', 'cabin_num'
                # 'last_name_count', 'is_alone_last_name'
                ] + \
                list(data.columns[data.columns.str.startswith('proba_by_')]) + \
                list(data.columns[data.columns.str.startswith('avg_expense_by_')]) + \
                list(data.columns[data.columns.str.startswith('homeplanet')]) + \
                list(data.columns[data.columns.str.startswith('destination')]) + \
                list(data.columns[data.columns.str.startswith('deck')]) + \
                list(data.columns[data.columns.str.startswith('side')])

In [116]:
a = set(data.columns)
b = set(feature_list)
a - b

{'Cabin',
 'Name',
 'PassengerId',
 'Transported',
 'dataset',
 'first_name',
 'last_name',
 'passenger_group'}

In [117]:
# Split train & test data
new_train_data = data[data['dataset'] == 'train'].copy()
new_test_data = data[data['dataset'] == 'test'].reset_index(drop=True).copy()

In [118]:
# Prepare features and target variable
X = new_train_data[feature_list]
y = new_train_data['Transported']

# Prepare test data
X_test = new_test_data[feature_list]

X.shape

(8693, 39)

## **Model Training**

In [119]:
# Initialize XGBoost parameters
# Use optuna to find the best hyperparameters
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_estimators': 800,
    'learning_rate': 0.02,
    'max_depth': 6,
    # 'n_estimators': 1200,
    # 'learning_rate': 0.11077380509652308,
    # 'max_depth': 5,
    # 'subsample': 0.9398024891956241,
    # 'colsample_bytree': 0.9223705385674421,
    # 'gamma': 0.5384219989060403,
    'random_state': 0,
    'early_stopping_rounds': 50
}

# Store results
metrics = {
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'roc_auc': []
}

# Training model
n_splits=5
kf = KFold(n_splits=n_splits)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\nFold {fold}/{n_splits}")

    # Split data
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Initialize and train model
    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50,

    )

    # Predictions
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    y_proba = y_proba[:, 1]

    # Calculate metrics
    metrics['accuracy'].append(accuracy_score(y_val, y_pred))
    metrics['precision'].append(precision_score(y_val, y_pred, average='weighted'))
    metrics['recall'].append(recall_score(y_val, y_pred, average='weighted'))
    metrics['f1'].append(f1_score(y_val, y_pred, average='weighted'))

    # ROC AUC (handle multi-class)
    if len(np.unique(y)) > 2:
        metrics['roc_auc'].append(roc_auc_score(y_val, y_proba, multi_class='ovo'))
    else:
        metrics['roc_auc'].append(roc_auc_score(y_val, y_proba))

# Print results
print("\nFinal Metrics Across All Folds:")
for metric, values in metrics.items():
    print(f"{metric.capitalize()}: {np.mean(values):.4f} ± {np.std(values):.4f}")


Fold 1/5
[0]	validation_0-logloss:0.68668
[50]	validation_0-logloss:0.50276
[100]	validation_0-logloss:0.45550
[150]	validation_0-logloss:0.43981
[200]	validation_0-logloss:0.43292
[250]	validation_0-logloss:0.42987
[300]	validation_0-logloss:0.42788
[350]	validation_0-logloss:0.42551
[400]	validation_0-logloss:0.42361
[450]	validation_0-logloss:0.42330
[497]	validation_0-logloss:0.42366

Fold 2/5
[0]	validation_0-logloss:0.68734
[50]	validation_0-logloss:0.49061
[100]	validation_0-logloss:0.44017
[150]	validation_0-logloss:0.42439
[200]	validation_0-logloss:0.41661
[250]	validation_0-logloss:0.41428
[300]	validation_0-logloss:0.41339
[350]	validation_0-logloss:0.41195
[400]	validation_0-logloss:0.41135
[450]	validation_0-logloss:0.41085
[500]	validation_0-logloss:0.40981
[550]	validation_0-logloss:0.40924
[600]	validation_0-logloss:0.40903
[650]	validation_0-logloss:0.40856
[700]	validation_0-logloss:0.40762
[750]	validation_0-logloss:0.40750
[799]	validation_0-logloss:0.40761

Fold 

In [120]:
# Create feature importance DataFrame with proper names
feature_importances = pd.DataFrame({
    'feature': feature_list,
    'importance': model.feature_importances_  # Or mean_importances from cross-val
}).sort_values('importance', ascending=False)

# Display with clean formatting
print("\nFeature Importances:")
print(feature_importances.to_string(index=False))


Feature Importances:
                             feature  importance
                      total_expenses    0.186776
                 proba_by_HomePlanet    0.152788
                           CryoSleep    0.075110
                 proba_by_cabin_deck    0.065920
           avg_expense_by_cabin_deck    0.058088
                        ShoppingMall    0.054551
                           FoodCourt    0.051934
                 proba_by_cabin_side    0.045989
                                 Spa    0.038108
                         RoomService    0.036026
                              VRDeck    0.034920
                              deck_F    0.024193
                              deck_C    0.023683
                           cabin_num    0.020257
                     homeplanet_Mars    0.019334
          avg_expense_by_Destination    0.015468
                proba_by_Destination    0.015142
      proba_by_passenger_group_count    0.015016
                                 VIP    0.01410

## **Prediction and Submission**

In [121]:
# Final model training
params = {
    'objective': 'binary:logistic',  # Change for multi-class
    'n_estimators': 800,
    'learning_rate': 0.02,
    'max_depth': 6,
    # 'n_estimators': 1200,
    # 'learning_rate': 0.11077380509652308,
    # 'max_depth': 5,
    # 'subsample': 0.9398024891956241,
    # 'colsample_bytree': 0.9223705385674421,
    # 'gamma': 0.5384219989060403,
    'random_state': 0
}

final_model = XGBClassifier(**params)
final_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=800, n_jobs=None,
              num_parallel_tree=None, random_state=0, ...)

In [109]:
# Prepare submission
test_preds = final_model.predict(X_test)

# Convert 0 or 1 into FALSE or TRUE
test_preds = test_preds.astype(bool)

submission_data = pd.DataFrame({
    'PassengerId': new_test_data['PassengerId'],
    'Transported': test_preds
})
submission_data.to_csv(dir_path+'final_submission_v13.csv', index=False)